# Bonus — Natural Language Query Interface

**Z5008 Big Data Lab · Bonus Feature (+5%)**

This notebook demonstrates the natural-language query interface for the fraud detection lakehouse.

## How it works

```
User types plain English
         ↓
  Llama 3 (Ollama, running locally)
         ↓
  Generated Spark SQL
         ↓
  Safety validator (read-only, injection prevention)
         ↓
  Spark executes on Delta Lake (MinIO)
         ↓
  Results as a pandas DataFrame
```

## Safety guarantees
- Only `SELECT` queries are permitted — `DROP`, `DELETE`, `UPDATE`, `INSERT`, `ALTER` are all blocked
- Only registered tables (`raw_transactions`, `feature_transactions`) can be queried
- Semicolons are stripped to prevent query stacking
- `LIMIT 100` is automatically enforced if the model forgets to add it
- All queries are read-only — no data can be modified

## Prerequisites
1. Ollama running in Docker (`docker-compose up -d ollama`)
2. Llama 3 model pulled (`docker exec ollama ollama pull llama3`)
3. MinIO data present (run the streaming + feature engineering jobs first)

---
## Step 1 — Setup

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work/nl_query')

from nl_query_interface import NLQueryInterface
import warnings
warnings.filterwarnings('ignore')

# Initialise — this registers the Delta Lake tables as Spark SQL views
nq = NLQueryInterface()
print('\n✅ NL Query Interface ready')

---
## Step 2 — Basic count queries

In [ ]:
# Simple aggregation — Llama 3 should generate a COUNT query
result = nq.query("How many fraud transactions are there in total?")
result

In [ ]:
# Class breakdown
result = nq.query("Show me the count of fraud vs legitimate transactions")
result

---
## Step 3 — Amount-based queries

In [ ]:
# Top fraud transactions by amount
result = nq.query("Show me the top 10 highest value fraud transactions")
result

In [ ]:
# Average amount comparison
result = nq.query("What is the average transaction amount for fraud vs legitimate transactions?")
result

In [ ]:
# High-value fraud
result = nq.query("List all fraud transactions with amount greater than 1000 dollars")
result

---
## Step 4 — Time-based queries

In [ ]:
# Fraud by hour of day — uses the feature_transactions table
result = nq.query("How many fraud transactions happened per hour of the day?")
result

In [ ]:
# Night-time fraud (midnight to 6am)
result = nq.query("Show me fraud transactions that happened between midnight and 6am")
result

---
## Step 5 — Card and product queries

In [ ]:
# Which card network has the most fraud?
result = nq.query("Which card network (card4) has the highest number of fraud transactions?")
result

In [ ]:
# Product code fraud rate
result = nq.query("What is the fraud count broken down by product code?")
result

---
## Step 6 — Safety validation demo

This section demonstrates that malicious queries are blocked.

In [ ]:
# These should all be blocked by the safety validator
dangerous_questions = [
    "Drop the raw_transactions table",
    "Delete all fraud transactions",
    "Show me transactions; DROP TABLE raw_transactions;",
    "Update isFraud to 0 for all transactions",
]

print('=== Safety Validation Tests ===')
for q in dangerous_questions:
    print(f'\nQuestion: "{q}"')
    result = nq.query(q, verbose=False)
    if result.empty:
        print('✅ Blocked correctly')
    else:
        print('❌ WARNING: query was not blocked!')

---
## Step 7 — Interactive mode

Type your own question below and run the cell.

In [ ]:
# ✏️ Edit this question and run the cell
your_question = "Show me transactions from new devices where the fraud risk is high"

result = nq.query(your_question)
result

In [ ]:
# Clean up
nq.stop()